In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

url = "https://www.transfermarkt.com.br/premier-league/spieltag/wettbewerb/GB1/saison_id/2025/spieltag/1"

response = requests.get(url, headers=headers)

response.status_code

200

In [3]:
soup = BeautifulSoup(response.content, "html.parser")

In [4]:
# Storing only the 10 matches from the round, while filtering and retaining only the relevant information.
all_matches = soup.find_all('table', {'style':'border-top: 0 !important;'})

display(all_matches)
display(len(all_matches))

[<table style="border-top: 0 !important;">
 <tbody>
 <tr class="table-grosse-schrift">
 <td class="rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname">
 <span class="tabellenplatz">(5.)</span> 
                                                                     <a href="/fc-liverpool/spielplan/verein/31/saison_id/2025" title="FC Liverpool">Liverpool</a> </td>
 <td class="rechts hauptlink no-border-rechts show-for-small spieltagsansicht-vereinsname">
 <span class="tabellenplatz">(5.)</span> 
                                                                     <a href="/fc-liverpool/spielplan/verein/31/saison_id/2025" title="FC Liverpool">LIV</a> </td>
 <td class="hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen">
 <a href="/fc-liverpool/spielplan/verein/31/saison_id/2025" title="FC Liverpool"><img alt="FC Liverpool" class="" src="https://tmssl.akamaized.net//images/wappen/small/31.png?lm=1727873452" title="FC Liverpo

10

In [5]:
# Testing where can I find the team names.
home_team = all_matches[0].find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'}).find('a').get('title')
away_team = all_matches[0].find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'}).find('a').get('title')

display(home_team)
display(away_team)

'FC Liverpool'

'AFC Bournemouth'

In [6]:
# Testing where can I find the critical information.
# When a match has no goals and no red cards, the "match" value is set to 0.

match_events = all_matches[1].find_all('tr', {'class':'no-border spieltagsansicht-aktionen'})

goal_player = match_events[0].find('a').get('title')
home_goal_minute = match_events[0].find('td', {'class':'zentriert no-border-links'}).string
away_goal_minute = match_events[0].find('td', {'class':'zentriert no-border-rechts'}).string

try: penalty = match_events[0].find('td', {'class':'rechts no-border-rechts spieltagsansicht'}).find_all('span')[2].get('class')[1]
except: penalty = match_events[0].find('td', {'class':'links no-border-links spieltagsansicht'}).find('span').get('class')[1]

display(goal_player)
display(home_goal_minute)
display(away_goal_minute)
display(penalty)

'Ezri Konsa'

"66'"

'\xa0'

'icon-rotekarte-formation'

In [9]:
# Displaying the scrapped data in a data frame

goals_list = []
count_goal = 0

for match in all_matches:

    event = match.find_all('tr', {'class':'no-border spieltagsansicht-aktionen'})

    # List with the entire class necesaire to get the home and away team's names
    gross_h_team = match.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
    gross_a_team = match.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

    # Checking for a possible forum buttom
    home_forum_check = gross_h_team.find('a').get('href')
    away_forum_check = gross_a_team.find('a').get('href')

    # Different ways to get the title depending if it has the forum buttom
    if 'forum' in home_forum_check and 'forum' in away_forum_check:
        h_team = gross_h_team.find_all('a')[1].get('title')
        a_team = gross_a_team.find_all('a')[1].get('title')
    elif 'forum' in home_forum_check:
        h_team = gross_h_team.find_all('a')[1].get('title')
        a_team = gross_a_team.find('a').get('title')
    elif 'forum' in away_forum_check:
        h_team = gross_h_team.find('a').get('title')
        a_team = gross_a_team.find_all('a')[1].get('title')
    else:
        h_team = gross_h_team.find('a').get('title')
        a_team = gross_a_team.find('a').get('title')

    for row in event:
        temp = []
        
        # Goal primary key
        count_goal += 1
        if count_goal < 10: goal_id = 'G-000' + str(count_goal)
        elif count_goal < 100: goal_id = 'G-00' + str(count_goal)
        elif count_goal < 1000: goal_id = 'G-0' + str(count_goal)
        else: count_goal = 'G-' + str(count_goal+1)
        temp.append(goal_id)

        # Home Team Events
        try: 
            event_type = row.find('td', {'class':'rechts no-border-rechts spieltagsansicht'}).find_all('span')[2].get('class')[1]
            goal_minute = row.find('td', {'class':'zentriert no-border-links'}).string
            temp.append(h_team)
            temp.append(goal_minute)
        
        # Away Team Events
        except: 
            event_type = row.find('td', {'class':'links no-border-links spieltagsansicht'}).find('span').get('class')[1]
            goal_minute = row.find('td', {'class':'zentriert no-border-rechts'}).string
            temp.append(a_team)
            temp.append(goal_minute)

        # Event Types Information
        if event_type == 'icon-tor-formation': temp.append(0) # Normal Goal
        elif event_type == 'icon-elfmeter-formation': temp.append(1) # Penalty Goal
        elif event_type == 'icon-eigentor-formation': temp.append(2) # Own Contra
        elif event_type == 'icon-verschossener-elfmeter-formation': temp.append(-2) # Penalty Missed
        else: temp.append(-1) # Red Cards

        # Player wich made the action
        player = row.find('a').get('title')
        temp.append(player)    

        goals_list.append(temp)

df_goals = pd.DataFrame(goals_list)
df_goals.columns = ['goal_id', 'goal_score_team','goal_minute','goal_type', 'goal_scorer_name']
display(df_goals)

,goal_id,goal_score_team,goal_minute,goal_type,goal_scorer_name
0,G-0001,FC Liverpool,37',0,Hugo Ekitiké
1,G-0002,FC Liverpool,49',0,Cody Gakpo
2,G-0003,AFC Bournemouth,64',0,Antoine Semenyo
3,G-0004,AFC Bournemouth,76',0,Antoine Semenyo
4,G-0005,FC Liverpool,88',0,Federico Chiesa
5,G-0006,FC Liverpool,90+4',0,Mohamed Salah
6,G-0007,Aston Villa FC,66',-1,Ezri Konsa
7,G-0008,Brighton & Hove Albion,55',1,Matt O'Riley
8,G-0009,FC Fulham,90+6',0,Rodrigo Muniz
9,G-0010,AFC Sunderland,61',0,Eliezer Mayenda
